# Basic Data Analysis of the NAture Milling 2025 Dataset

In [ ]:
# Imports
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets
import os
import glob
from tqdm import tqdm

#### Load csv with pandas and display it

In [ ]:
# Load metadata and measurement files, transform to single DataFrame

metadata_path = 'metadata.csv'
#data_folder = '../data/00_NASA_Milling'  # Adjust if needed
data_folder='/Users/flore/Downloads/archive'
metadata = pd.read_csv(metadata_path,sep=';')[['ExperimentIndex','CycleToFailureNormalized']]
def get_measurement_files(folder):
    return sorted(glob.glob(os.path.join(folder, 'P*_F*_C*')))
def extract_signals(df):
    # vib_spindle: 3 spindle axes
    spindle = df[['Accelerometer - Spindle +Y','Accelerometer - Spindle -Z','Accelerometer - Spindle -X']].values
    vib_spindle = np.linalg.norm(spindle, axis=1)
    # vib_table: 5 table axes
    table = df[['Accelerometer - X Driving axle +Z','Accelerometer - X Driving axle -X','Accelerometer - Y Driving axle +Z','Accelerometer - Y Driving axle +Y','Accelerometer - Y Driving axle -X']].values
    vib_table = np.linalg.norm(table, axis=1)
    return vib_spindle, vib_table
records = []
for idx, row in tqdm(metadata.iterrows(), total=metadata.shape[0]):
    fname = os.path.join(data_folder, row['ExperimentIndex']+'.csv')
    if not os.path.exists(fname):
        continue
    df = pd.read_csv(fname)
    vib_spindle, vib_table = extract_signals(df)
    records.append({
        'ExperimentIndex': row['ExperimentIndex'],
        'CycleToFailureNormalized': row['CycleToFailureNormalized'],
        'vib_spindle': vib_spindle,
        'vib_table': vib_table
    })
final_df = pd.DataFrame(records)
final_df.head()

In [ ]:
metadata.head()

In [ ]:
# Rename P to CaseID and F to layer and C to cycle
final_df[['CaseID', 'layer', 'cycle']] = final_df['ExperimentIndex'].str.extract(r'P(\d+)_F(\d+)_C(\d+)').astype(int)
final_df.head()

In [ ]:
# Create a 'run' column that joins 'layer' and 'cycle' for unique run numbering per case
final_df = final_df.sort_values(['CaseID', 'layer', 'cycle'])
final_df['run'] = final_df.groupby('CaseID').cumcount() + 1
final_df.head(20)

In [ ]:
# rename CycleToFailureNormalized to wear_norm
final_df=final_df.rename(columns={'CycleToFailureNormalized':'wear_norm'})

In [ ]:
# Transform the wear_norm values from comma separated value into a float
final_df['wear_norm'] = final_df['wear_norm'].astype(str).str.replace(',', '.').astype(float)
final_df['wear_norm'].head()

## Homogenization

In [ ]:
# invert the wear
final_df['wear_norm'] = 1 - final_df['wear_norm']
final_df['wear_norm'].head()

In [ ]:
# LEts remove the outlier cases CaseID 26,110 
final_df_cleaned = final_df[~final_df['CaseID'].isin([7,16,26,41,110,114])]

In [ ]:
#Visualize the wear ('VB') over time ('run') for each case ('case')
import seaborn as sns
plt.figure(figsize=(12, 6))
sns.lineplot(data=final_df_cleaned, x='run', y='wear_norm', hue='CaseID', marker='o',palette='tab20')
plt.title('Tool Wear Over Time for Each Case')
plt.xlabel('Cut Number')
plt.ylabel('Normalized Tool Wear (1 = No Wear, 0 = Max Wear)')
plt.legend(title='Case ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Find CaseIDs with a negative slope for wear_norm over run
from scipy.stats import linregress
negative_slope_cases = []
for case_id, group in final_df_cleaned.groupby('CaseID'):
    slope, _, _, _, _ = linregress(group['run'], group['wear_norm'])
    if slope < 0:
        negative_slope_cases.append(case_id)
print('CaseIDs with negative wear slope:', negative_slope_cases)

In [ ]:
# save to parquet
final_df.to_parquet('nature_milling_2025_normalized.parquet', index=False)

In [ ]:
# Typical signal length
lengths = final_df['vib_spindle'].apply(len)
print(f"Typical signal length: {lengths.mode()[0]} samples")

In [ ]:
# visualize a few example signals
import random
example_cases = random.sample(final_df['CaseID'].unique().tolist(), 3)
plt.figure(figsize=(15, 10))
for i, case_id in enumerate(example_cases):
    case_data = final_df[final_df['CaseID'] == case_id].sort_values('run')
    plt.subplot(3, 1, i+1)
    for _, row in case_data.iterrows():
        plt.plot(row['vib_spindle'], alpha=0.3)
    plt.title(f'CaseID {case_id} - Spindle Vibration Signals')
    plt.xlabel('Sample Index')
    plt.ylabel('Vibration Magnitude')
plt.tight_layout()
plt.show()

In [ ]:
# unique case ids
print('Unique CaseIDs:', final_df['CaseID'].unique())